# Historical HealthKit Data Demo

This notebook demonstrates how to retrieve historical HealthKit quantity data from Google Cloud Storage using the `get_historic_hk_quantity()` method.

## Overview

Historical HealthKit data is stored in GCS as zstd-compressed JSON files at:
```
{storage_bucket}/users/{user_id}/historicalHealthSamples/{HK_identifier}_{UUID}.json.zstd
```

The `get_historic_hk_quantity()` method:
- Lists all matching files for a given user and observation type
- Downloads and decompresses each file
- Parses FHIR observations into a pandas DataFrame
- Returns the same 17-column structure as `get_hk_quantity()`

In [ ]:
from datetime import datetime

from myheartcounts_ds import MHC4Client, MHCConfig, DiscoveredObservationType

In [ ]:
# Initialize client (defaults to production)
client = MHC4Client()

print(f"Project: {client.config.project_id}")
print(f"Storage Bucket: {client.config.storage_bucket}")

## Get a User ID

First, let's get a user ID to query historical data for.

In [ ]:
# Get some users to work with
users = client.list_users(limit=5)
print(f"Found {len(users)} users")

for user in users:
    print(f"  {user.id} - enrolled: {user.date_of_enrollment}")

In [ ]:
# Select a user for the demo
user_id = users[0].id if users else "example-user-id"
print(f"Using user: {user_id}")

## Retrieve Historical Heart Rate Data

Use `get_historic_hk_quantity()` to fetch historical HealthKit quantity observations from GCS.

In [ ]:
# Fetch all historical heart rate data for the user
df_historic = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id=user_id,
)

print(f"Historical heart rate records: {len(df_historic)}")
print(f"\nDataFrame columns ({len(df_historic.columns)}):")
for col in df_historic.columns:
    print(f"  - {col}")

In [ ]:
# Display first few records
if not df_historic.empty:
    print("First 5 records:")
    display(df_historic.head())
else:
    print("No historical data found for this user.")

In [ ]:
# Summary statistics
if not df_historic.empty:
    print(f"Date range: {df_historic['start_time'].min()} to {df_historic['start_time'].max()}")
    print(f"\nHeart rate statistics:")
    print(f"  Min: {df_historic['value'].min():.1f} {df_historic['unit'].iloc[0]}")
    print(f"  Max: {df_historic['value'].max():.1f} {df_historic['unit'].iloc[0]}")
    print(f"  Mean: {df_historic['value'].mean():.1f} {df_historic['unit'].iloc[0]}")
    print(f"  Median: {df_historic['value'].median():.1f} {df_historic['unit'].iloc[0]}")
    
    # Unique sources
    sources = df_historic['source_name'].dropna().unique()
    print(f"\nData sources ({len(sources)}):")
    for source in sources:
        count = len(df_historic[df_historic['source_name'] == source])
        print(f"  - {source}: {count:,} records")

## Time Filtering

Filter historical data by time range using `start_time` and `end_time` parameters.

In [ ]:
# Get historical data for a specific time range
df_filtered = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id=user_id,
    start_time=datetime(2024, 1, 1),
    end_time=datetime(2024, 7, 1),
)

print(f"Historical heart rate records (Jan-Jun 2024): {len(df_filtered)}")

if not df_filtered.empty:
    print(f"Date range: {df_filtered['start_time'].min()} to {df_filtered['start_time'].max()}")

## Compare Historical vs Real-time Data

Compare data from GCS (historical) with data from Firestore (real-time).

In [ ]:
# Get real-time data from Firestore
df_realtime = client.get_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id=user_id,
)

# Get historical data from GCS
df_historic = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id=user_id,
)

print(f"Data comparison for user {user_id}:")
print(f"  Firestore (real-time): {len(df_realtime):,} records")
print(f"  GCS (historical): {len(df_historic):,} records")

In [ ]:
df_historic

In [ ]:
# Compare date ranges
if not df_realtime.empty:
    print(f"\nFirestore date range:")
    print(f"  {df_realtime['start_time'].min()} to {df_realtime['start_time'].max()}")

if not df_historic.empty:
    print(f"\nGCS date range:")
    print(f"  {df_historic['start_time'].min()} to {df_historic['start_time'].max()}")

## Other Historical Quantity Types

The method works with any `HK_QUANTITY_*` observation type.

In [ ]:
# Historical step count data
df_steps = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_STEP_COUNT,
    user_id=user_id,
)

print(f"Historical step count records: {len(df_steps)}")

if not df_steps.empty:
    print(f"Date range: {df_steps['start_time'].min()} to {df_steps['start_time'].max()}")
    print(f"Total steps: {df_steps['value'].sum():,.0f}")

In [ ]:
df_steps

In [ ]:
# Historical active energy burned
df_energy = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_ACTIVE_ENERGY_BURNED,
    user_id=user_id,
)

print(f"Historical active energy records: {len(df_energy)}")

if not df_energy.empty:
    print(f"Date range: {df_energy['start_time'].min()} to {df_energy['start_time'].max()}")
    print(f"Total energy: {df_energy['value'].sum():,.0f} {df_energy['unit'].iloc[0]}")

## List Available Historic Observation Types

Use `list_historic_observation_types()` and `list_historic_hk_quantity_observation_types()` to discover what observation types are available in a user's historical GCS data.

**Note:** Unlike the Firestore listing methods, these methods only support single-user queries (no sampling across users).

In [ ]:
# List all historic observation types for this user
all_historic_types = client.list_historic_observation_types(user_id=user_id)
print(f"All historic observation types ({len(all_historic_types)}):")
for t in sorted(all_historic_types):
    print(f"  - {t}")

In [ ]:
# List only HK quantity types
hk_quantity_types = client.list_historic_hk_quantity_observation_types(user_id=user_id)
print(f"Historic HK quantity types ({len(hk_quantity_types)}):")
for t in sorted(hk_quantity_types):
    print(f"  - {t}")

## Error Handling

The method handles errors gracefully:
- Returns empty DataFrame if no files exist
- Skips files that fail to download/parse (logs warning)
- Raises `ValueError` for invalid observation types

In [ ]:
# Empty result for user with no historical data
df_empty = client.get_historic_hk_quantity(
    DiscoveredObservationType.HK_QUANTITY_HEART_RATE,
    user_id="nonexistent-user-id",
)

print(f"Records for nonexistent user: {len(df_empty)}")
print(f"DataFrame has correct schema: {list(df_empty.columns)[:5]}...")